# Additive (Bahdanau) attention from scratch

Run the cells top to bottom. Every step prints its result, and the numbers are small enough to check by hand.

**The setup.** You have
- a **decoder state** (the "query"): what the decoder is thinking right now, shape `(decoder_dim,)`
- **encoder states** (the "keys" and "values"): one vector per input position, shape `(seq_len, encoder_dim)`

**The question attention answers:** how much should each of the `seq_len` input positions matter right now?

**The recipe (3 steps):**
1. **Score** every position: $\text{score}_i = v^\top \tanh(W\,\text{decoder\_state} + U\,\text{encoder\_state}_i)$
2. **Normalize** with softmax, so the weights are positive and sum to 1
3. **Blend**: the context vector is the weighted average of the encoder states

"Additive" refers to the `+` inside the tanh: query and key are projected and *added*. Dot-product attention multiplies them instead.

### Names in this notebook vs. symbols in the paper

| This notebook | Bahdanau et al. 2014 | Shape | What it is |
|---|---|---|---|
| `decoder_state` | $s_{i-1}$ | `(decoder_dim,)` | the query |
| `encoder_states` | $h_1 \dots h_T$ | `(seq_len, encoder_dim)` | one vector per input position |
| `query_weights` | $W_a$ | `(attn_dim, decoder_dim)` | projects the query into "attention space" |
| `key_weights` | $U_a$ | `(attn_dim, encoder_dim)` | projects each encoder state into attention space |
| `score_vector` | $v_a$ | `(attn_dim,)` | collapses each vector to one number |
| `scores` | $e_{ij}$ | `(seq_len,)` | one raw score per position |
| `attention_weights` | $\alpha_{ij}$ | `(seq_len,)` | scores after softmax, sum to 1 |
| `context_vector` | $c_i$ | `(encoder_dim,)` | the weighted average, the output |

`attn_dim` is a size you choose. Both inputs get projected to it so they can be added.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 0. What does `@` do?

`@` is **matrix multiplication**. (In NumPy it is `np.matmul`, and PyTorch tensors use the same symbol.) It is *not* the same as `*`, which multiplies element by element.

The rule: the **last dimension of the left side must equal the first (or only) dimension of the right side**. Those matching "inner" dimensions get multiplied and summed away, and the outer ones remain.

`(3, 2) @ (2,)  ->  (3,)`   the 2s match and disappear.

In [2]:
matrix = np.array([[1., 2.],
                   [3., 4.],
                   [5., 6.]])          # shape (3, 2)
vector = np.array([10., 20.])          # shape (2,)

# matrix @ vector: each ROW of the matrix is dotted with the vector
print("matrix @ vector      :", matrix @ vector)        # [1*10+2*20, 3*10+4*20, 5*10+6*20]

# vector @ vector: the ordinary dot product -> a single number
print("[1,2,3] @ [4,5,6]    :", np.array([1., 2., 3.]) @ np.array([4., 5., 6.]))   # 1*4+2*5+3*6

# matrix @ matrix: (3,2) @ (2,3) -> (3,3)
print("matrix @ matrix.T    :\n", matrix @ matrix.T)

# * is element by element, NOT matrix multiplication
print("vector * [3, 4]      :", vector * np.array([3., 4.]))

# the inner dimensions must match
try:
    matrix @ np.array([1., 2., 3.])    # (3,2) @ (3,): 2 != 3
except ValueError as error:
    print("ValueError:", error)

matrix @ vector      : [ 50. 110. 170.]
[1,2,3] @ [4,5,6]    : 32.0
matrix @ matrix.T    :
 [[ 5. 11. 17.]
 [11. 25. 39.]
 [17. 39. 61.]]
vector * [3, 4]      : [30. 80.]
ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 2)


`.T` transposes (swaps rows and columns), so `(3, 2).T` becomes `(2, 3)`. We use it below because the encoder states are stored **one per row**, `(seq_len, encoder_dim)`. To project all of them at once we write `encoder_states @ key_weights.T`, which is `(seq_len, encoder_dim) @ (encoder_dim, attn_dim) -> (seq_len, attn_dim)`.

## 1. Softmax (the only helper we need)

$\mathrm{softmax}(x)_i = \dfrac{e^{x_i}}{\sum_j e^{x_j}}$

**Trick:** subtract the max before exponentiating. Softmax gives the same answer after subtracting a constant (it cancels top and bottom), but `exp` no longer overflows on large numbers.

In [3]:
def softmax(values, axis=-1):
    shifted_values = values - values.max(axis=axis, keepdims=True)   # same result, no overflow
    exp_values = np.exp(shifted_values)
    return exp_values / exp_values.sum(axis=axis, keepdims=True)

print(softmax(np.array([1.0, 2.0, 3.0])))
print(softmax(np.array([1001.0, 1002.0, 1003.0])))   # same answer; a naive exp(1001) would overflow

[0.09   0.2447 0.6652]
[0.09   0.2447 0.6652]


## 2. A tiny example you can verify by hand

3 input positions, `encoder_dim = decoder_dim = attn_dim = 2`. I use identity matrices for the two weight matrices and `[1, 1]` for `score_vector` so the arithmetic is easy to follow. In a real model these are learned.

In [4]:
decoder_state = np.array([1.0, 0.0])         # the query, shape (2,)
encoder_states = np.array([[1.0, 0.0],       # position 1
                           [0.0, 1.0],       # position 2
                           [1.0, 1.0]])      # position 3      shape (seq_len=3, encoder_dim=2)

query_weights = np.eye(2)                    # (attn_dim, decoder_dim)
key_weights = np.eye(2)                      # (attn_dim, encoder_dim)
score_vector = np.array([1.0, 1.0])          # (attn_dim,)

## 3. The forward pass, one line at a time

**Step 1a: project the query.** `query_weights @ decoder_state` is `(attn_dim, decoder_dim) @ (decoder_dim,) -> (attn_dim,)`. With identity weights it equals `[1, 0]`.

In [5]:
projected_query = query_weights @ decoder_state
print("projected_query =", projected_query, " shape", projected_query.shape)

projected_query = [1. 0.]  shape (2,)


**Step 1b: project every encoder state at once.** `(seq_len, encoder_dim) @ (encoder_dim, attn_dim) -> (seq_len, attn_dim)`. Row `i` is the projection of position `i`.

In [6]:
projected_keys = encoder_states @ key_weights.T
print("projected_keys =\n", projected_keys, "\nshape", projected_keys.shape)

projected_keys =
 [[1. 0.]
 [0. 1.]
 [1. 1.]] 
shape (3, 2)


**Step 1c: add, then tanh.** `projected_query + projected_keys` adds the length-2 query to *each row* of the `(3, 2)` matrix. NumPy does this automatically ("broadcasting"), and this is the "additive" step.

Check position 1 by hand: `[1,0] + [1,0] = [2,0]`, then `tanh(2) = 0.9640` and `tanh(0) = 0`.

In [7]:
summed = projected_query + projected_keys
print("projected_query + projected_keys =\n", summed)
activated = np.tanh(summed)
print("after tanh =\n", activated)

projected_query + projected_keys =
 [[2. 0.]
 [1. 1.]
 [2. 1.]]
after tanh =
 [[0.964  0.    ]
 [0.7616 0.7616]
 [0.964  0.7616]]


**Step 1d: collapse each row into one score with `score_vector`.** `(seq_len, attn_dim) @ (attn_dim,) -> (seq_len,)`: each row is dotted with `score_vector`, giving one number per position.

Position 1: `0.9640*1 + 0*1 = 0.9640`. Position 2: `0.7616 + 0.7616 = 1.5232`. Position 3: `0.9640 + 0.7616 = 1.7256`.

In [8]:
scores = activated @ score_vector
print("scores =", scores)

scores = [0.964  1.5232 1.7256]


**Step 2: softmax turns scores into weights.** Position 3 scored highest, so it gets the most weight. The weights sum to 1.

In [9]:
attention_weights = softmax(scores)
print("attention_weights =", attention_weights, " sum =", attention_weights.sum())

attention_weights = [0.2045 0.3576 0.4379]  sum = 1.0


**Step 3: weighted average of the encoder states.** `(seq_len,) @ (seq_len, encoder_dim) -> (encoder_dim,)`: each encoder state is scaled by its weight and they are added up.

By hand: `0.2045*[1,0] + 0.3576*[0,1] + 0.4379*[1,1] = [0.6424, 0.7955]`.

In [10]:
context_vector = attention_weights @ encoder_states
print("context_vector =", context_vector)

context_vector = [0.6424 0.7955]


## 4. The same thing as one function

In [11]:
def additive_attention(decoder_state, encoder_states, query_weights, key_weights, score_vector):
    projected_query = query_weights @ decoder_state            # (attn_dim,)
    projected_keys = encoder_states @ key_weights.T            # (seq_len, attn_dim)
    scores = np.tanh(projected_query + projected_keys) @ score_vector   # (seq_len,)
    attention_weights = softmax(scores)                        # (seq_len,)
    context_vector = attention_weights @ encoder_states        # (encoder_dim,)
    return context_vector, attention_weights, scores

result_context, result_weights, result_scores = additive_attention(
    decoder_state, encoder_states, query_weights, key_weights, score_vector)
assert np.allclose(context_vector, result_context) and np.allclose(attention_weights, result_weights)
print("function matches the step-by-step version")

function matches the step-by-step version


## 5. Why the tanh matters (worth being able to explain)

Suppose we drop it. The score becomes $v^\top(W\,\text{decoder\_state} + U\,\text{encoder\_state}_i) = \underbrace{v^\top W\,\text{decoder\_state}}_{\text{same for every position } i} + v^\top U\,\text{encoder\_state}_i$.

The query term is a constant added to *every* score, and softmax ignores constant shifts, so **the query would have no effect on the weights at all**. The tanh mixes query and position non-linearly *before* `score_vector` collapses them, so the score can depend on the pair. Test it:

In [12]:
def scores_without_tanh(decoder_state, encoder_states, query_weights, key_weights, score_vector):
    return (query_weights @ decoder_state + encoder_states @ key_weights.T) @ score_vector

other_decoder_state = np.array([-5.0, 3.0])      # a very different query
args = (encoder_states, query_weights, key_weights, score_vector)
print("no tanh,   query 1:", softmax(scores_without_tanh(decoder_state, *args)))
print("no tanh,   query 2:", softmax(scores_without_tanh(other_decoder_state, *args)))   # identical weights
print("with tanh, query 1:", additive_attention(decoder_state, *args)[1])
print("with tanh, query 2:", additive_attention(other_decoder_state, *args)[1])          # weights change

no tanh,   query 1: [0.2119 0.2119 0.5761]
no tanh,   query 2: [0.2119 0.2119 0.5761]
with tanh, query 1: [0.2045 0.3576 0.4379]
with tanh, query 2: [0.3324 0.3337 0.3339]


## 6. Batched version with a padding mask

Real inputs come in batches, and sequences in a batch have different lengths, so shorter ones get padded.

Shapes: `decoder_states (batch, decoder_dim)`, `encoder_states (batch, seq_len, encoder_dim)`, `padding_mask (batch, seq_len)` with `True` for real tokens.

Two things to get right:
- **Broadcasting:** `projected_query` is `(batch, attn_dim)` but `projected_keys` is `(batch, seq_len, attn_dim)`. `projected_query[:, None, :]` inserts a length-1 axis to make `(batch, 1, attn_dim)`, which broadcasts across `seq_len`. (PyTorch calls this `.unsqueeze(1)`.)
- **Masking:** set padded scores to `-inf` **before** the softmax. Then `exp(-inf) = 0`, so padded positions get exactly zero weight and the real ones still sum to 1.

In [13]:
def additive_attention_batched(decoder_states, encoder_states, padding_mask,
                               query_weights, key_weights, score_vector):
    projected_query = decoder_states @ query_weights.T                   # (batch, attn_dim)
    projected_keys = encoder_states @ key_weights.T                      # (batch, seq_len, attn_dim)
    scores = np.tanh(projected_query[:, None, :] + projected_keys) @ score_vector   # (batch, seq_len)
    scores = np.where(padding_mask, scores, -np.inf)                     # mask BEFORE softmax
    attention_weights = softmax(scores, axis=1)                          # softmax over seq_len
    context_vector = (attention_weights[:, :, None] * encoder_states).sum(axis=1)   # (batch, encoder_dim)
    return context_vector, attention_weights

decoder_states = np.stack([decoder_state, decoder_state])      # batch of 2
batched_encoder_states = np.stack([encoder_states, encoder_states])
padding_mask = np.array([[True, True, True],
                         [True, True, False]])                 # sequence 2 has only 2 real tokens

batched_context, batched_weights = additive_attention_batched(
    decoder_states, batched_encoder_states, padding_mask, query_weights, key_weights, score_vector)
print("attention_weights:\n", batched_weights)
print("context_vector:\n", batched_context)

attention_weights:
 [[0.2045 0.3576 0.4379]
 [0.3637 0.6363 0.    ]]
context_vector:
 [[0.6424 0.7955]
 [0.3637 0.6363]]


Row 1 matches the single-query result. In row 2 the padded position has weight `0.0`, and the other two are `softmax([0.9640, 1.5232]) = [0.3637, 0.6363]`.

In [14]:
assert np.allclose(batched_context[0], context_vector)
assert np.allclose(batched_weights.sum(axis=1), 1.0)
assert batched_weights[1, 2] == 0.0
print("all checks passed")

all checks passed


## 7. The same thing in PyTorch

PyTorch tensors work like NumPy arrays, and `@`, `.T` and broadcasting behave the same way. The differences you need right now:

| NumPy | PyTorch |
|---|---|
| `np.array(...)` | `torch.tensor(...)` |
| `np.tanh(x)` | `torch.tanh(x)` |
| softmax by hand | `torch.softmax(x, dim=...)` (`dim` plays the role of `axis`) |
| `x[:, None, :]` | `x.unsqueeze(1)` |
| `np.where(mask, x, -np.inf)` | `x.masked_fill(~mask, float("-inf"))` (`~` flips True and False) |

First, the single-query function, translated line by line:

In [15]:
import torch
import torch.nn as nn

def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32)

def additive_attention_torch(decoder_state, encoder_states, query_weights, key_weights, score_vector):
    projected_query = query_weights @ decoder_state
    projected_keys = encoder_states @ key_weights.T
    scores = torch.tanh(projected_query + projected_keys) @ score_vector
    attention_weights = torch.softmax(scores, dim=-1)
    context_vector = attention_weights @ encoder_states
    return context_vector, attention_weights, scores

torch_context, torch_weights, torch_scores = additive_attention_torch(
    to_tensor(decoder_state), to_tensor(encoder_states),
    to_tensor(query_weights), to_tensor(key_weights), to_tensor(score_vector))
print("torch scores  :", torch_scores)
print("torch weights :", torch_weights)
print("torch context :", torch_context)
assert np.allclose(torch_context.numpy(), context_vector, atol=1e-6)

torch scores  : tensor([0.9640, 1.5232, 1.7256])
torch weights : tensor([0.2045, 0.3576, 0.4379])
torch context : tensor([0.6424, 0.7955])


Second, the version you would actually use in a model: an `nn.Module`. It stores the three weight matrices as `nn.Linear` layers, which PyTorch can then train.

`nn.Linear(in_features, out_features, bias=False)` holds a weight of shape `(out_features, in_features)` and computes `x @ weight.T`. That is exactly the `@ weights.T` pattern we have been writing, which is why our weight shapes are `(attn_dim, input_dim)`.

In [16]:
class AdditiveAttention(nn.Module):
    def __init__(self, decoder_dim, encoder_dim, attn_dim):
        super().__init__()
        self.query_projection = nn.Linear(decoder_dim, attn_dim, bias=False)   # holds query_weights
        self.key_projection = nn.Linear(encoder_dim, attn_dim, bias=False)     # holds key_weights
        self.score_projection = nn.Linear(attn_dim, 1, bias=False)             # holds score_vector

    def forward(self, decoder_states, encoder_states, padding_mask):
        projected_query = self.query_projection(decoder_states)                # (batch, attn_dim)
        projected_keys = self.key_projection(encoder_states)                   # (batch, seq_len, attn_dim)
        hidden = torch.tanh(projected_query.unsqueeze(1) + projected_keys)     # (batch, seq_len, attn_dim)
        scores = self.score_projection(hidden).squeeze(-1)                     # (batch, seq_len)
        scores = scores.masked_fill(~padding_mask, float("-inf"))
        attention_weights = torch.softmax(scores, dim=1)
        context_vector = (attention_weights.unsqueeze(-1) * encoder_states).sum(dim=1)
        return context_vector, attention_weights

attention_layer = AdditiveAttention(decoder_dim=2, encoder_dim=2, attn_dim=2)

# load the SAME weights we used in NumPy so the outputs are comparable
with torch.no_grad():
    attention_layer.query_projection.weight.copy_(to_tensor(query_weights))
    attention_layer.key_projection.weight.copy_(to_tensor(key_weights))
    attention_layer.score_projection.weight.copy_(to_tensor(score_vector).unsqueeze(0))   # (1, attn_dim)

torch_batched_context, torch_batched_weights = attention_layer(
    to_tensor(decoder_states), to_tensor(batched_encoder_states), torch.tensor(padding_mask))
print("module weights:\n", torch_batched_weights)
assert np.allclose(torch_batched_weights.detach().numpy(), batched_weights, atol=1e-6)
assert np.allclose(torch_batched_context.detach().numpy(), batched_context, atol=1e-6)
print("PyTorch module matches the NumPy version")

module weights:
 tensor([[0.2045, 0.3576, 0.4379],
        [0.3637, 0.6363, 0.0000]], grad_fn=<SoftmaxBackward0>)
PyTorch module matches the NumPy version
